In [ ]:
import pandas as pd
import re
from sentence_transformers import SentenceTransformer
from sklearn.cluster import DBSCAN

# Dados de exemplo: cada linha representa um item com suas características
data = [
    ["MOBILIÁRIOS", "7105", "MOBILIÁRIO DOMÉSTICO", "323", "ARMÁRIO COPA/COZINHA", "461685",
     "ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA MDP, TIPO:BALCÃO, ACABAMENTO SUPERFICIAL:ENVERNIZADO, QUANTIDADE PORTAS:2 UN, LARGURA:1,20 M, PROFUNDIDADE:0,30 M, ALTURA:0,70 M, CARACTERÍSTICAS ADICIONAIS:COM 4 GAVETAS, SEM TAMPO"],
    ["MOBILIÁRIOS", "7105", "MOBILIÁRIO DOMÉSTICO", "323", "ARMÁRIO COPA/COZINHA", "464315",
     "ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA DE DEMOLIÇÃO, TIPO:LOUCEIRO, ACABAMENTO SUPERFICIAL:PINTADO, QUANTIDADE PRATELEIRAS:3 UN, LARGURA:0,90 M, PROFUNDIDADE:0,40 M, ALTURA:1,80 M"],
    ["MOBILIÁRIOS", "7105", "MOBILIÁRIO DOMÉSTICO", "323", "ARMÁRIO COPA/COZINHA", "473251",
     "ARMÁRIO COPA/COZINHA, MATERIAL:AÇO INOXIDÁVEL, TIPO:BALCÃO, ACABAMENTO SUPERFICIAL:LISO, QUANTIDADE PORTAS:2 UN, QUANTIDADE PRATELEIRAS:1 UN, LARGURA:1,50 M, PROFUNDIDADE:0,70 M, ALTURA:0,85 M, CARACTERÍSTICAS ADICIONAIS:PORTAS CORREDIÇAS"],
    ["MOBILIÁRIOS", "7105", "MOBILIÁRIO DOMÉSTICO", "325", "CABIDE GUARDA-ROUPA", "600948",
     "CABIDE GUARDA-ROUPA, MATERIAL CORPO:PVC, MATERIAL GANCHO:PVC, COR:PRETA, CARACTERÍSTICAS ADICIONAIS:PONTAS ARREDONDADAS"],
    ["MOBILIÁRIOS", "7105", "MOBILIÁRIO DOMÉSTICO", "329", "BIOMBO", "207093",
     "BIOMBO, MATERIAL:MADEIRA, QUANTIDADE MÓDULOS:3 UN, LARGURA MÓDULO:40 CM, ESPESSURA MÓDULO:2 CM, ALTURA MÓDULO:1,70 CM"]
]

colunas = ["Categoria1", "Codigo1", "Categoria2", "Codigo2", "Nome", "CodigoItem", "Descricao"]
df = pd.DataFrame(data, columns=colunas)

# Função para limpar e normalizar a descrição
def clean_description(desc):
    # Converter para minúsculas
    desc = desc.lower()
    # Remover caracteres especiais, mantendo letras, números e espaços
    desc = re.sub(r'[^a-z0-9áàâãéèêíïóôõöúç\s]', ' ', desc)
    # Remover espaços duplicados
    desc = re.sub(r'\s+', ' ', desc).strip()
    return desc

# Aplica a limpeza nas descrições
df['clean_desc'] = df['Descricao'].apply(clean_description)

# Carrega o modelo para gerar embeddings (aqui usamos o modelo "paraphrase-MiniLM-L6-v2")
model = SentenceTransformer('paraphrase-MiniLM-L6-v2')
embeddings = model.encode(df['clean_desc'].tolist())

# Clusterização utilizando DBSCAN com métrica de similaridade cosseno
# O parâmetro eps pode ser ajustado conforme a similaridade desejada entre itens
clustering_model = DBSCAN(eps=0.8, min_samples=1, metric='cosine')
cluster_labels = clustering_model.fit_predict(embeddings)
df['cluster'] = cluster_labels

# Exibe os itens com seus respectivos grupos
print("Itens agrupados:")
print(df[['CodigoItem', 'Descricao', 'cluster']])


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.51k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Itens agrupados:
  CodigoItem                                          Descricao  cluster
0     461685  ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA MDP, TI...        0
1     464315  ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA DE DEMO...        0
2     473251  ARMÁRIO COPA/COZINHA, MATERIAL:AÇO INOXIDÁVEL,...        0
3     600948  CABIDE GUARDA-ROUPA, MATERIAL CORPO:PVC, MATER...        0
4     207093  BIOMBO, MATERIAL:MADEIRA, QUANTIDADE MÓDULOS:3...        0


In [ ]:
%pip install nltk -q
%pip install scikit-learn --upgrade -q


In [ ]:
import pandas as pd
import numpy as np
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
import re

# Instalar recursos necessários do NLTK
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Classe para processamento e agrupamento de itens de licitação
class ClassificadorItensLicitacao:
    def __init__(self):
        self.stop_words = set(stopwords.words('portuguese'))
        self.vectorizer = TfidfVectorizer()

    def _preprocessamento(self, texto):
        """Preprocessa o texto removendo caracteres especiais e stopwords"""
        texto = texto.lower()
        texto = re.sub(r'[^\w\s]', ' ', texto)
        tokens = word_tokenize(texto)
        tokens = [w for w in tokens if not w in self.stop_words and len(w) > 2]
        return ' '.join(tokens)

    def _parse_item(self, texto_completo):
        """Extrai características estruturadas do item"""
        linhas = texto_completo.strip().split()

        # Identificar padrões nas linhas de descrição
        categoria = linhas[0] if len(linhas) > 0 else ""
        codigo_categoria = linhas[1] if len(linhas) > 1 else ""
        subcategoria = linhas[2] if len(linhas) > 2 else ""
        codigo_subcategoria = linhas[3] if len(linhas) > 3 else ""
        tipo_item = linhas[4] if len(linhas) > 4 else ""
        codigo_item = linhas[5] if len(linhas) > 5 else ""

        # Juntar o restante como descrição completa
        descricao = ' '.join(linhas[6:]) if len(linhas) > 6 else ""

        # Extrair atributos específicos da descrição
        atributos = {}
        if "MATERIAL:" in descricao:
            match = re.search(r'MATERIAL:([^,]+)', descricao)
            if match:
                atributos['material'] = match.group(1).strip()

        if "TIPO:" in descricao:
            match = re.search(r'TIPO:([^,]+)', descricao)
            if match:
                atributos['tipo'] = match.group(1).strip()

        return {
            'categoria': categoria,
            'codigo_categoria': codigo_categoria,
            'subcategoria': subcategoria,
            'codigo_subcategoria': codigo_subcategoria,
            'tipo_item': tipo_item,
            'codigo_item': codigo_item,
            'descricao_completa': descricao,
            'descricao_processada': self._preprocessamento(descricao),
            'atributos': atributos,
            'texto_original': texto_completo
        }

    def processar_itens(self, lista_textos):
        """Processa uma lista de descrições de itens"""
        self.itens = [self._parse_item(texto) for texto in lista_textos]
        self.df_itens = pd.DataFrame([{
            'id': idx,
            'categoria': item['categoria'],
            'subcategoria': item['subcategoria'],
            'tipo_item': item['tipo_item'],
            'codigo_item': item['codigo_item'],
            'descricao_processada': item['descricao_processada'],
            'texto_original': item['texto_original']
        } for idx, item in enumerate(self.itens)])

        return self.df_itens

    def calcular_similaridade(self):
        """Calcula matriz de similaridade entre os itens"""
        # Vetorizar as descrições
        X = self.vectorizer.fit_transform(self.df_itens['descricao_processada'])

        # Calcular similaridade entre documentos
        self.matriz_similaridade = cosine_similarity(X)

        return self.matriz_similaridade

    def agrupar_itens(self, limiar_similaridade=0.3, n_clusters=None):
        """Agrupa itens semelhantes usando clustering hierárquico"""
        if not hasattr(self, 'matriz_similaridade'):
            self.calcular_similaridade()

        # Distância = 1 - similaridade
        matriz_distancia = 1 - self.matriz_similaridade

        # Aplicar clustering hierárquico
        if n_clusters:
            clustering = AgglomerativeClustering(
                n_clusters=n_clusters,
                metric='precomputed',
                linkage='average'
            )
        else:
            clustering = AgglomerativeClustering(
                distance_threshold=limiar_similaridade,
                n_clusters=None,
                metric='precomputed',
                linkage='average'
            )

        self.df_itens['grupo'] = clustering.fit_predict(matriz_distancia)

        return self.df_itens

    def gerar_relatorio_grupos(self):
        """Gera um relatório dos grupos de itens para licitação"""
        if not 'grupo' in self.df_itens.columns:
            raise ValueError("Primeiro execute agrupar_itens() para classificar os itens")

        relatorio = []
        for grupo in sorted(self.df_itens['grupo'].unique()):
            itens_grupo = self.df_itens[self.df_itens['grupo'] == grupo]

            # Determinar características comuns do grupo
            tipos = itens_grupo['tipo_item'].unique()
            categorias = itens_grupo['categoria'].unique()
            subcategorias = itens_grupo['subcategoria'].unique()

            relatorio.append({
                'grupo_id': grupo,
                'qtd_itens': len(itens_grupo),
                'tipos': tipos,
                'categorias': categorias,
                'subcategorias': subcategorias,
                'codigos_itens': itens_grupo['codigo_item'].tolist(),
                'itens': itens_grupo['texto_original'].tolist()
            })

        return relatorio

# Exemplo de uso com os dados fornecidos
def main():
    # Lista de descrições de itens
    itens = [
        "MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 323 ARMÁRIO COPA/COZINHA 461685 ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA MDP, TIPO:BALCÃO, ACABAMENTO SUPERFICIAL:ENVERNIZADO, QUANTIDADE PORTAS:2 UN, LARGURA:1,20 M, PROFUNDIDADE:0,30 M, ALTURA:0,70 M, CARACTERÍSTICAS ADICIONAIS:COM 4 GAVETAS, SEM TAMPO",
        "MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 323 ARMÁRIO COPA/COZINHA 464315 ARMÁRIO COPA/COZINHA, MATERIAL:MADEIRA DE DEMOLIÇÃO, TIPO:LOUCEIRO, ACABAMENTO SUPERFICIAL:PINTADO, QUANTIDADE PRATELEIRAS:3 UN, LARGURA:0,90 M, PROFUNDIDADE:0,40 M, ALTURA:1,80 M",
        "MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 323 ARMÁRIO COPA/COZINHA 473251 ARMÁRIO COPA/COZINHA, MATERIAL:AÇO INOXIDÁVEL, TIPO:BALCÃO, ACABAMENTO SUPERFICIAL:LISO, QUANTIDADE PORTAS:2 UN, QUANTIDADE PRATELEIRAS:1 UN, LARGURA:1,50 M, PROFUNDIDADE:0,70 M, ALTURA:0,85 M, CARACTERÍSTICAS ADICIONAIS:PORTAS CORREDIÇAS",
        "MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 325 CABIDE GUARDA-ROUPA 600948 CABIDE GUARDA-ROUPA, MATERIAL CORPO:PVC, MATERIAL GANCHO:PVC, COR:PRETA, CARACTERÍSTICAS ADICIONAIS:PONTAS ARREDONDADAS",
        "MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 329 BIOMBO 207093 BIOMBO, MATERIAL:MADEIRA, QUANTIDADE MÓDULOS:3 UN, LARGURA MÓDULO:40 CM, ESPESSURA MÓDULO:2 CM, ALTURA MÓDULO:1,70 CM"
    ]

    # Inicializar e usar o classificador
    classificador = ClassificadorItensLicitacao()
    df_itens = classificador.processar_itens(itens)

    print("Itens processados:")
    print(df_itens[['id', 'categoria', 'subcategoria', 'tipo_item']])

    # Calcular similaridade
    matriz_sim = classificador.calcular_similaridade()
    print("\nMatriz de similaridade:")
    print(matriz_sim)

    # Agrupar itens
    df_agrupado = classificador.agrupar_itens(limiar_similaridade=0.7)
    print("\nItens agrupados:")
    print(df_agrupado[['id', 'tipo_item', 'grupo']])

    # Gerar relatório
    relatorio = classificador.gerar_relatorio_grupos()

    print("\nRelatório de grupos para licitação:")
    for grupo in relatorio:
        print(f"\nGrupo {grupo['grupo_id']} - {len(grupo['itens'])} itens")
        print(f"Tipos de itens: {', '.join(grupo['tipos'])}")
        print(f"Categorias: {', '.join(grupo['categorias'])}")
        print(f"Subcategorias: {', '.join(grupo['subcategorias'])}")
        print(f"Códigos dos itens: {', '.join(grupo['codigos_itens'])}")
        print("Itens neste grupo:")
        for i, item in enumerate(grupo['itens']):
            print(f"  {i+1}. {item[:100]}...")

if __name__ == "__main__":
    main()

Itens processados:
   id    categoria subcategoria tipo_item
0   0  MOBILIÁRIOS   MOBILIÁRIO       323
1   1  MOBILIÁRIOS   MOBILIÁRIO       323
2   2  MOBILIÁRIOS   MOBILIÁRIO       323
3   3  MOBILIÁRIOS   MOBILIÁRIO       325
4   4  MOBILIÁRIOS   MOBILIÁRIO       329

Matriz de similaridade:
[[1.         0.55983587 0.63069437 0.07557764 0.1111673 ]
 [0.55983587 1.         0.54396203 0.0279463  0.1223048 ]
 [0.63069437 0.54396203 1.         0.06865023 0.0928442 ]
 [0.07557764 0.0279463  0.06865023 1.         0.02542705]
 [0.1111673  0.1223048  0.0928442  0.02542705 1.        ]]

Itens agrupados:
   id tipo_item  grupo
0   0       323      0
1   1       323      0
2   2       323      0
3   3       325      1
4   4       329      2

Relatório de grupos para licitação:

Grupo 0 - 3 itens
Tipos de itens: 323
Categorias: MOBILIÁRIOS
Subcategorias: MOBILIÁRIO
Códigos dos itens: ARMÁRIO, ARMÁRIO, ARMÁRIO
Itens neste grupo:
  1. MOBILIÁRIOS 7105 MOBILIÁRIO DOMÉSTICO 323 ARMÁRIO COPA/COZINHA

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
